# Object Detection Training Template with Optuna

이 노트북은 다음 전제를 기준으로 작성되어 있습니다.

- 객체 박스는 `annotations.points` 에서 읽습니다.
- 객체 클래스는 `annotations.disease` 값을 사용합니다.
- 한 이미지에 박스가 여러 개 있을 수 있습니다.
- train / val 데이터는 폴더로 직접 분리합니다.
- 증강 데이터는 원본과 함께 그대로 학습에 포함합니다.

## 데이터 형식
라벨은 이미지마다 JSON 파일 1개입니다.

- 원본 JSON: `description.image`, `annotations.disease`, `annotations.points`
- 증강 JSON: 위 필드 + `description.original`, `augmented`

증강 여부와 상관없이 이 노트북은 동일한 방식으로 파싱합니다.

## 워크플로우
1. 경로와 실행 옵션 설정
2. JSON 라벨을 읽어 데이터셋 생성
3. Optuna로 백본/하이퍼파라미터 탐색
4. 베스트 파라미터로 최종 학습
5. 샘플 추론 확인

## 현재 최적화 기준
추가 의존성 없이 바로 돌리기 위해 Optuna 목적함수는 `validation loss` 최소화 기준으로 구성했습니다.

In [ ]:
import copy
import json
import random
import zipfile
from dataclasses import dataclass
from pathlib import Path

import optuna
import torch
import torchvision
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import boxes as box_ops

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('optuna:', optuna.__version__)

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 0  # Windows에서는 0 또는 2 권장
PIN_MEMORY = torch.cuda.is_available()

# =========================
# 1) AIHub 압축 데이터 경로 설정
# =========================
AIHUB_ROOT = Path('data/071.시설 작물 질병 진단/01.데이터')
TRAIN_RAW_ROOT = AIHUB_ROOT / '1.Training'
VAL_RAW_ROOT = AIHUB_ROOT / '2.Validation'

EXTRACT_ROOT = Path('data/_extracted_071')
AUTO_EXTRACT_ZIP = True


def _zip_stem(zip_path: Path):
    return zip_path.name[:-4] if zip_path.name.lower().endswith('.zip') else zip_path.stem


def _extract_zip_once(zip_path: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    done_flag = out_dir / '.done'
    if done_flag.exists():
        return False

    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(out_dir)

    done_flag.write_text('ok', encoding='utf-8')
    return True


def extract_split_archives(split_raw_root: Path, split_name: str):
    label_zip_root = split_raw_root / '라벨링데이터' / '04.딸기'
    image_zip_root = split_raw_root / '원천데이터' / '04.딸기'

    if not label_zip_root.exists() or not image_zip_root.exists():
        raise FileNotFoundError(f'압축 폴더를 찾을 수 없습니다: {split_raw_root}')

    split_extract_root = EXTRACT_ROOT / split_name
    image_out_root = split_extract_root / 'images'
    label_out_root = split_extract_root / 'labels'

    image_zip_files = sorted(image_zip_root.glob('*.zip'))
    label_zip_files = sorted(label_zip_root.glob('*.zip'))

    extracted_count = 0

    for zip_path in image_zip_files:
        extracted_count += int(_extract_zip_once(zip_path, image_out_root / _zip_stem(zip_path)))

    for zip_path in label_zip_files:
        extracted_count += int(_extract_zip_once(zip_path, label_out_root / _zip_stem(zip_path)))

    print(f'[{split_name}] image_zips={len(image_zip_files)} label_zips={len(label_zip_files)} newly_extracted={extracted_count}')

    return {
        'name': f'{split_name}_all',
        'image_dir': image_out_root,
        'label_dir': label_out_root,
    }


def build_sources(auto_extract: bool = True):
    if auto_extract:
        train_source = extract_split_archives(TRAIN_RAW_ROOT, 'train')
        val_source = extract_split_archives(VAL_RAW_ROOT, 'val')
    else:
        train_source = {
            'name': 'train_all',
            'image_dir': EXTRACT_ROOT / 'train' / 'images',
            'label_dir': EXTRACT_ROOT / 'train' / 'labels',
        }
        val_source = {
            'name': 'val_all',
            'image_dir': EXTRACT_ROOT / 'val' / 'images',
            'label_dir': EXTRACT_ROOT / 'val' / 'labels',
        }

    return [train_source], [val_source]


TRAIN_SOURCES, VAL_SOURCES = build_sources(auto_extract=AUTO_EXTRACT_ZIP)

# disease id -> 표시 이름. 비워두면 disease_숫자 형태로 자동 생성됩니다.
CLASS_NAME_BY_DISEASE = {
    # 0: 'normal',
    # 7: 'powdery_mildew',
}

# =========================
# 2) Optuna / 학습 설정
# =========================
SEARCH_BACKBONES = ['resnet50', 'convnext_tiny']
SEARCH_TRIALS = 12
SEARCH_EPOCHS = 3
FINAL_EPOCHS = 15
BATCH_SIZE_OPTIONS = [2, 4]
IMAGE_MIN_SIZE = 640
IMAGE_MAX_SIZE = 640

STUDY_NAME = 'strawberry_detection_optuna'
OUTPUT_DIR = Path('runs') / STUDY_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STUDY_DB_PATH = OUTPUT_DIR / 'optuna_study.db'

print('device:', DEVICE)
print('train_sources:', TRAIN_SOURCES)
print('val_sources:', VAL_SOURCES)
print('output:', OUTPUT_DIR.resolve())

torch: 2.11.0+cu130
torchvision: 0.26.0+cu130
optuna: 4.8.0
device: cuda
output: C:\potenup3\pj03-SFAX\runs\strawberry_detection_optuna


## 현재 데이터 구조 반영 방식

현재 코드는 아래처럼 압축 파일 구조를 직접 읽도록 맞춰져 있습니다.

```text
data/
  071.시설 작물 질병 진단/
    01.데이터/
      1.Training/
        원천데이터/04.딸기/*.zip
        라벨링데이터/04.딸기/*.zip
      2.Validation/
        원천데이터/04.딸기/*.zip
        라벨링데이터/04.딸기/*.zip
```

설정 셀에서 하는 일

- zip 파일을 `data/_extracted_071/train` 과 `data/_extracted_071/val` 로 자동 해제
- 이미지 zip / 라벨 zip 을 각각 모두 합쳐서 train source 1개, val source 1개로 구성
- 다음 실행부터는 `.done` 파일이 있으면 재해제하지 않음

주의

- 첫 실행 시 압축 해제 시간이 걸릴 수 있습니다.
- 압축을 다시 풀고 싶으면 `data/_extracted_071` 폴더를 지운 뒤 다시 실행하세요.
- 매칭은 JSON 안의 `description.image` 값을 기준으로 수행됩니다.

In [ ]:
# =========================
# 3) JSON 파서 / 데이터셋
# =========================
@dataclass
class Record:
    image_path: Path
    label_path: Path
    disease_id: int
    boxes: list
    source_name: str
    is_augmented: bool
    original_name: str | None


def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_json(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def list_json_files(label_dir: Path):
    if not label_dir.exists():
        return []
    return sorted(label_dir.rglob('*.json'))


def build_image_index(image_dir: Path):
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    image_index = {}
    duplicate_count = 0

    for image_path in image_dir.rglob('*'):
        if image_path.is_file() and image_path.suffix.lower() in exts:
            key = image_path.name
            if key in image_index:
                duplicate_count += 1
                continue
            image_index[key] = image_path

    if duplicate_count:
        print(f'warning: duplicated image names ignored={duplicate_count}')

    return image_index


def parse_record(label_path: Path, image_index: dict, source_name: str):
    data = load_json(label_path)
    description = data.get('description', {})
    annotations = data.get('annotations', {})
    points = annotations.get('points', []) or []
    disease_id = int(annotations.get('disease', 0))
    image_name = description.get('image')

    if not image_name:
        return None

    image_path = image_index.get(image_name)
    if image_path is None:
        return None

    boxes = []
    for point in points:
        xtl = float(point['xtl'])
        ytl = float(point['ytl'])
        xbr = float(point['xbr'])
        ybr = float(point['ybr'])
        if xbr > xtl and ybr > ytl:
            boxes.append([xtl, ytl, xbr, ybr])

    return Record(
        image_path=image_path,
        label_path=label_path,
        disease_id=disease_id,
        boxes=boxes,
        source_name=source_name,
        is_augmented='augmented' in data,
        original_name=description.get('original'),
    )


def collect_records(sources):
    records = []
    missing_images = []

    for source in sources:
        image_dir = Path(source['image_dir'])
        label_dir = Path(source['label_dir'])
        source_name = source['name']

        image_index = build_image_index(image_dir)
        json_files = list_json_files(label_dir)

        for label_path in json_files:
            record = parse_record(label_path, image_index=image_index, source_name=source_name)
            if record is None:
                missing_images.append(str(label_path))
                continue
            records.append(record)

        source_usable = sum(1 for record in records if record.source_name == source_name)
        print(f'{source_name}: images={len(image_index)}, labels={len(json_files)}, usable={source_usable}')

    if missing_images:
        print(f'missing image references: {len(missing_images)}')
        print('first 5 missing examples:')
        for item in missing_images[:5]:
            print(' -', item)

    return records


def build_class_mapping(records, class_name_by_disease):
    disease_ids = sorted({record.disease_id for record in records})
    disease_to_label = {disease_id: idx + 1 for idx, disease_id in enumerate(disease_ids)}
    label_to_name = {
        label: class_name_by_disease.get(disease_id, f'disease_{disease_id}')
        for disease_id, label in disease_to_label.items()
    }
    return disease_to_label, label_to_name


class JsonDetectionDataset(Dataset):
    def __init__(self, records, disease_to_label):
        self.records = records
        self.disease_to_label = disease_to_label

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        image = Image.open(record.image_path).convert('RGB')
        width, height = image.size

        boxes = torch.tensor(record.boxes, dtype=torch.float32) if record.boxes else torch.zeros((0, 4), dtype=torch.float32)
        if len(boxes):
            boxes = box_ops.clip_boxes_to_image(boxes, (height, width))

        label_value = self.disease_to_label[record.disease_id]
        labels = torch.full((len(boxes),), label_value, dtype=torch.int64) if len(boxes) else torch.zeros((0,), dtype=torch.int64)

        image_tensor = torchvision.transforms.functional.to_tensor(image)
        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]) if len(boxes) else torch.zeros((0,), dtype=torch.float32)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx], dtype=torch.int64),
            'area': area,
            'iscrowd': torch.zeros((len(labels),), dtype=torch.int64),
        }
        return image_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


def prepare_datasets():
    train_records = collect_records(TRAIN_SOURCES)
    val_records = collect_records(VAL_SOURCES)

    if not train_records:
        raise ValueError('No train records found. TRAIN_SOURCES 경로를 확인하세요.')
    if not val_records:
        raise ValueError('No validation records found. VAL_SOURCES 경로를 확인하세요.')

    disease_to_label, label_to_name = build_class_mapping(train_records + val_records, CLASS_NAME_BY_DISEASE)
    train_dataset = JsonDetectionDataset(train_records, disease_to_label)
    val_dataset = JsonDetectionDataset(val_records, disease_to_label)

    print('train samples:', len(train_dataset))
    print('val samples:', len(val_dataset))
    print('classes:', {label: name for label, name in sorted(label_to_name.items())})

    return train_dataset, val_dataset, disease_to_label, label_to_name

In [5]:
# =========================
# 4) 모델 / 학습 함수
# =========================
def build_resnet50_fasterrcnn(num_classes: int):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights='DEFAULT',
        trainable_backbone_layers=3,
        min_size=IMAGE_MIN_SIZE,
        max_size=IMAGE_MAX_SIZE,
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


def build_convnext_tiny_fasterrcnn(num_classes: int):
    from torchvision.models import convnext_tiny
    from torchvision.models.detection.backbone_utils import BackboneWithFPN

    backbone_body = convnext_tiny(weights='DEFAULT').features
    backbone = BackboneWithFPN(
        backbone=backbone_body,
        return_layers={'2': '0', '4': '1', '6': '2'},
        in_channels_list=[192, 384, 768],
        out_channels=256,
    )

    anchor_generator = AnchorGenerator(
        sizes=((32,), (64,), (128,), (256,), (512,)),
        aspect_ratios=((0.5, 1.0, 2.0),) * 5,
    )

    roi_pooler = torchvision.ops.MultiScaleRoIAlign(
        featmap_names=['0', '1', '2', 'pool'],
        output_size=7,
        sampling_ratio=2,
    )

    model = FasterRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=roi_pooler,
        min_size=IMAGE_MIN_SIZE,
        max_size=IMAGE_MAX_SIZE,
    )
    return model


def build_model(backbone_name: str, num_classes: int):
    if backbone_name == 'resnet50':
        return build_resnet50_fasterrcnn(num_classes)
    if backbone_name == 'convnext_tiny':
        return build_convnext_tiny_fasterrcnn(num_classes)
    raise ValueError(f'Unsupported backbone: {backbone_name}')


def make_loaders(train_dataset, val_dataset, batch_size):
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_fn,
    )
    return train_loader, val_loader


def train_one_epoch(model, data_loader, optimizer, device, epoch):
    model.train()
    running_loss = 0.0

    for step, (images, targets) in enumerate(data_loader, start=1):
        images = [image.to(device) for image in images]
        targets = [{k: v.to(device) for k, v in target.items()} for target in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        running_loss += float(loss.item())

        if step % 10 == 0 or step == len(data_loader):
            print(f'[epoch {epoch:02d}] step {step:04d}/{len(data_loader):04d} loss={loss.item():.4f}')

    return running_loss / max(len(data_loader), 1)


def evaluate_val_loss(model, data_loader, device):
    model.train()
    total_loss = 0.0
    with torch.no_grad():
        for images, targets in data_loader:
            images = [image.to(device) for image in images]
            targets = [{k: v.to(device) for k, v in target.items()} for target in targets]
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
            total_loss += float(loss.item())
    return total_loss / max(len(data_loader), 1)


def run_training(model, train_loader, val_loader, optimizer, scheduler, device, epochs, trial=None):
    history = []
    best_state = None
    best_val_loss = float('inf')

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device, epoch)
        val_loss = evaluate_val_loss(model, val_loader, device)
        if scheduler is not None:
            scheduler.step()

        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})
        print(f'[epoch {epoch:02d}] train={train_loss:.4f} val={val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        if trial is not None:
            trial.report(best_val_loss, step=epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

    return history, best_state, best_val_loss

In [ ]:
# =========================
# 5) Optuna 탐색 실행
# =========================
set_seed(SEED)
train_dataset, val_dataset, disease_to_label, label_to_name = prepare_datasets()
num_classes = len(label_to_name) + 1


def objective(trial):
    backbone = trial.suggest_categorical('backbone', SEARCH_BACKBONES)
    batch_size = trial.suggest_categorical('batch_size', BATCH_SIZE_OPTIONS)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 5e-4, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)

    model = build_model(backbone, num_classes=num_classes).to(DEVICE)
    train_loader, val_loader = make_loaders(train_dataset, val_dataset, batch_size=batch_size)

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SEARCH_EPOCHS)

    _, _, best_val_loss = run_training(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=DEVICE,
        epochs=SEARCH_EPOCHS,
        trial=trial,
    )

    return best_val_loss


storage_url = f"sqlite:///{STUDY_DB_PATH.resolve().as_posix()}"
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=storage_url,
    direction='minimize',
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1),
)

study.optimize(objective, n_trials=SEARCH_TRIALS, show_progress_bar=True)

print('best value:', study.best_value)
print('best params:', study.best_trial.params)
study.best_trial.params

In [ ]:
# =========================
# 6) 베스트 파라미터로 최종 학습 + 저장 + 샘플 추론
# =========================
import matplotlib.pyplot as plt

best_params = study.best_trial.params
print('using params:', best_params)

final_model = build_model(best_params['backbone'], num_classes=num_classes).to(DEVICE)
final_train_loader, final_val_loader = make_loaders(
    train_dataset,
    val_dataset,
    batch_size=best_params['batch_size'],
)

final_optimizer = torch.optim.AdamW(
    final_model.parameters(),
    lr=best_params['learning_rate'],
    weight_decay=best_params['weight_decay'],
)
final_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(final_optimizer, T_max=FINAL_EPOCHS)

history, best_state, best_val_loss = run_training(
    model=final_model,
    train_loader=final_train_loader,
    val_loader=final_val_loader,
    optimizer=final_optimizer,
    scheduler=final_scheduler,
    device=DEVICE,
    epochs=FINAL_EPOCHS,
)

if best_state is not None:
    final_model.load_state_dict(best_state)

checkpoint = {
    'model_state_dict': final_model.state_dict(),
    'best_params': best_params,
    'best_val_loss': best_val_loss,
    'class_names': label_to_name,
    'disease_to_label': disease_to_label,
}

best_model_path = OUTPUT_DIR / 'best_model.pt'
last_model_path = OUTPUT_DIR / 'last_model.pt'
torch.save(checkpoint, best_model_path)
torch.save({**checkpoint, 'history': history}, last_model_path)

print('best validation loss:', best_val_loss)
print('saved best model to:', best_model_path.resolve())
print('saved last model to:', last_model_path.resolve())


@torch.inference_mode()
def predict_one(model, image_path: Path, score_threshold=0.3):
    model.eval()
    image = Image.open(image_path).convert('RGB')
    tensor = torchvision.transforms.functional.to_tensor(image).to(DEVICE)
    output = model([tensor])[0]

    scores = output['scores'].detach().cpu()
    boxes = output['boxes'].detach().cpu()
    labels = output['labels'].detach().cpu()
    keep = scores >= score_threshold

    return image, boxes[keep], labels[keep], scores[keep]


def show_prediction(image, boxes, labels, scores, label_to_name):
    canvas = image.copy()
    draw = ImageDraw.Draw(canvas)
    for box, label, score in zip(boxes.tolist(), labels.tolist(), scores.tolist()):
        x1, y1, x2, y2 = box
        class_name = label_to_name.get(label, f'class_{label}')
        draw.rectangle([x1, y1, x2, y2], outline='red', width=4)
        draw.text((x1, max(0, y1 - 20)), f'{class_name}: {score:.2f}', fill='red')

    plt.figure(figsize=(8, 8))
    plt.imshow(canvas)
    plt.axis('off')
    plt.show()


sample_record = val_dataset.records[0]
image, boxes, labels, scores = predict_one(final_model, sample_record.image_path, score_threshold=0.25)
print('sample image:', sample_record.image_path)
print('detections:', len(boxes))
show_prediction(image, boxes, labels, scores, label_to_name)